# 04 KG 경로 특징 추가 효과

무엇을 하는가: 외부 Reactome membership으로 계산한 proxy가 실제 분류 예측에 영향을 주는지 비교한다. 앞 노트북의 PCA/scGPT smoke 결과를 입력으로 사용한다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
assert (DATA / 'annotation/Ensembl2Reactome_All_Levels.human.tsv').exists()
run_script('05_kg_features_ablation.py', 'tutorial_kg_smoke.yaml')

/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/05_kg_features_ablation.py --config /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_kg_smoke.yaml


{"run_id": "tutorial_kg_smoke_pathway", "macro_f1": 0.17857142857142858, "balanced_accuracy": 0.25, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.1, "selected_class_weight": "balanced", "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.0}
{"run_id": "tutorial_kg_smoke_foundation_pathway", "macro_f1": 0.623015873015873, "balanced_accuracy": 0.6666666666666666, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.06666666666666667}
{"run_id": "tutorial_kg_smoke_foundation_random", "macro_f1": 0.8277777777777778, "balanced_accuracy": 0.8333333333333334, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight"

CompletedProcess(args=['/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python', '-u', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/05_kg_features_ablation.py', '--config', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_kg_smoke.yaml'], returncode=0, stdout='{"run_id": "tutorial_kg_smoke_pathway", "macro_f1": 0.17857142857142858, "balanced_accuracy": 0.25, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.1, "selected_class_weight": "balanced", "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.0}\n{"run_id": "tutorial_kg_smoke_foundation_pathway", "macro_f1": 0.623015873015873, "balanced_accuracy": 0.6666666666666666, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes

In [3]:
print(pd.read_csv(ROOT / 'reports/kg_ablation_smoke.tsv', sep='\t').to_string(index=False))
selection = json.loads((ROOT / 'results/tutorial_kg_smoke/selection.json').read_text())
print('selected pathways=', selection['selected_n'])
print('random sizes/degrees preserved=',selection['size_preserved'],selection['gene_membership_degree_preserved'])

                              run_id  macro_f1  balanced_accuracy  selected_C selected_class_weight  n_changed_test_predictions_vs_foundation  delta_macro_f1_vs_foundation
           tutorial_kg_smoke_pathway  0.178571           0.250000        0.10              balanced                                       7.0                     -0.588095
tutorial_kg_smoke_foundation_pathway  0.623016           0.666667        0.01                   NaN                                       2.0                     -0.143651
 tutorial_kg_smoke_foundation_random  0.827778           0.833333        0.01                   NaN                                       2.0                      0.061111
                  tutorial_pca_smoke  0.694444           0.750000        0.01              balanced                                       NaN                           NaN
                tutorial_scgpt_smoke  0.766667           0.833333        0.01                   NaN                                       Na

생물학적 해석: 경로 membership은 세포에서의 활성화·인과관계가 아니다. mean train-zscore proxy를 예측 입력으로 concatenate한다. pathway-only, embedding+pathway, embedding+무작위 대조를 동일 split/head로 비교한다. 무작위 대조는 경로 크기와 유전자 참여 횟수를 보존한다. 단순 결과 설명을 붙인 것과 예측 입력으로 사용한 것을 구분한다.

실패 해결: feature 파일이 없으면 02를 먼저 실행한다. 경로 수와 규제 강도는 validation만 사용한다. 성능 향상이 없으면 실제 결과 그대로 기록한다.